In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.get(scope="cnxsqlcontoso", key="sqlcnxcontoso")

In [0]:
conn_str = dbutils.secrets.get(
    scope="cnxsqlcontoso",
    key="sqlcnxcontoso"
)

parts = {}
for item in conn_str.split(";"):
    if "=" in item:
        k, v = item.split("=", 1)
        parts[k.strip()] = v.strip()

server = parts["Server"].replace("tcp:", "").split(",")[0]
database = parts["Initial Catalog"]
user = parts["User ID"]
password = parts["Password"]

jdbc_url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"database={database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

df_adresse= (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "[SalesLT].[Address]")
    .option("user", user)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(df_adresse)

In [0]:
from datetime import datetime

run_date = datetime.now().strftime("%Y%m%d_%H%M%S")

(df_adresse.write
    .mode("overwrite")
    .option("header", "true")
    .csv(
        f"abfss://salesdata@contososalesdata.dfs.core.windows.net/landing/adresse_{run_date}"
    ))

In [0]:
tables_df = (
    spark.read
         .format("jdbc")
         .option("url", jdbc_url)
         .option(
             "query",
             """
             SELECT TABLE_SCHEMA, TABLE_NAME
             FROM INFORMATION_SCHEMA.TABLES
             WHERE TABLE_TYPE = 'BASE TABLE'
             """
         )
         .option("user", user)
         .option("password", password)
         .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
         .load()
)

display(tables_df)

In [0]:
from datetime import datetime

run_date = datetime.now().strftime("%Y%m%d_%H%M%S")

tables = tables_df.collect()

for t in tables:

    schema_name = t["TABLE_SCHEMA"]
    table_name = t["TABLE_NAME"]

    full_table = f"{schema_name}.{table_name}"

    print(f"Export : {full_table}")

    df = (
        spark.read
             .format("jdbc")
             .option("url", jdbc_url)
             .option("dbtable", full_table)
             .option("user", user)
             .option("password", password)
             .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
             .load()
    )

    target_path = (
        f"abfss://salesdata@contososalesdata.dfs.core.windows.net/"
        f"landing/{table_name}/{run_date}"
    )

    (df.write
       .mode("overwrite")
       .option("header", "true")
       .csv(target_path))

In [0]:
print('hello')

In [0]:
print('bonjour')